### 公式化

token_t ~ P(token_t | token_1, token_2, ..., token_{t-1})

### 直观理解

基于模型的概率分布进行 **随机采样**

## 关键参数

### temperature

logits / temperature, 然后 softmax

### top-k sampling

限制候选的 token 数量

### top-p sampling

限制候选 token 的总概率

PS: 很明显, top-k 和 top-p 有些冲突, 这里我们只限制 temperature 和 top-k

In [67]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen3-0.6B'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
)
model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

### 取样演示

In [86]:
TEMPERATURE = 2
TOPK = 50
import torch 
import torch.nn.functional as F
# 取样方法
messages = [
    {'role': 'user', 'content': 'introduce yourself in 10 words'}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    enable_thinking = False,
    add_generation_prompt = True
)
inputs = tokenizer(
    text,
    return_tensors = 'pt'
)
outputs = model(**inputs)


In [87]:
outputs

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 3.5938,  3.7812,  3.6562,  ...,  1.6484,  1.6484,  1.6484],
         [ 7.2812,  8.2500,  6.3125,  ...,  0.4141,  0.4141,  0.4141],
         [ 4.7188,  9.6250, 11.5625,  ...,  3.8281,  3.8281,  3.8281],
         ...,
         [ 5.1250, 17.0000,  6.9375,  ..., -0.3418, -0.3418, -0.3418],
         [-4.0312, -2.5469, -7.7812,  ..., -1.3516, -1.3516, -1.3516],
         [ 7.4062, 12.5625,  5.7188,  ...,  4.2812,  4.2812,  4.2812]]],
       dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), hidden_states=None, atten

In [88]:
outputs_logits = outputs.logits
outputs_logits = outputs_logits[:, -1, :]
outputs_logits

tensor([[ 7.4062, 12.5625,  5.7188,  ...,  4.2812,  4.2812,  4.2812]],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [89]:
# temperature 处理
outputs_logits_temperature = outputs_logits / TEMPERATURE
outputs_logits_temperature

tensor([[3.7031, 6.2812, 2.8594,  ..., 2.1406, 2.1406, 2.1406]],
       dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [90]:
# top-k
outputs_logits_temperature_topk, _ = torch.topk(input=outputs_logits_temperature, k=TOPK, dim=-1)
outputs_logits_temperature_topk

tensor([[12.9375, 12.6875, 12.1875, 11.0625,  9.7500,  9.5000,  9.1250,  9.0000,
          8.8750,  8.8125,  8.8125,  8.8125,  8.6875,  8.3750,  8.3125,  8.3125,
          8.0625,  7.9688,  7.9688,  7.9688,  7.9062,  7.8438,  7.7812,  7.7812,
          7.7812,  7.7500,  7.7500,  7.7500,  7.7188,  7.7188,  7.6562,  7.6250,
          7.5938,  7.5625,  7.5312,  7.5000,  7.4688,  7.4375,  7.4375,  7.4062,
          7.4062,  7.4062,  7.4062,  7.3438,  7.3125,  7.2812,  7.2500,  7.2500,
          7.2500,  7.1875]], dtype=torch.bfloat16, grad_fn=<TopkBackward0>)

In [91]:
outputs_logits_temperature[outputs_logits_temperature < outputs_logits_temperature_topk[0][-1]] = - float('inf')
outputs_logits_temperature

tensor([[-inf, -inf, -inf,  ..., -inf, -inf, -inf]], dtype=torch.bfloat16,
       grad_fn=<IndexPutBackward0>)

In [92]:
# softmax
prob = F.softmax(input=outputs_logits_temperature_topk)
prob

/var/folders/1k/c9npdpq131sc3pgvycnb0ts80000gn/T/ipykernel_14271/2315111842.py:2: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  prob = F.softmax(input=outputs_logits_temperature_topk)


tensor([[0.3574, 0.2793, 0.1689, 0.0549, 0.0148, 0.0115, 0.0079, 0.0070, 0.0062,
         0.0058, 0.0058, 0.0058, 0.0051, 0.0037, 0.0035, 0.0035, 0.0027, 0.0025,
         0.0025, 0.0025, 0.0023, 0.0022, 0.0021, 0.0021, 0.0021, 0.0020, 0.0020,
         0.0020, 0.0019, 0.0019, 0.0018, 0.0018, 0.0017, 0.0017, 0.0016, 0.0016,
         0.0015, 0.0015, 0.0015, 0.0014, 0.0014, 0.0014, 0.0014, 0.0013, 0.0013,
         0.0013, 0.0012, 0.0012, 0.0012, 0.0011]], dtype=torch.bfloat16,
       grad_fn=<SoftmaxBackward0>)

In [111]:
# sample
next_token = torch.multinomial(input=prob, num_samples=1)
tokenizer.decode(next_token)

['!']

In [ ]:
import torch
import torch.nn.functional as F
# 采样解码
def sample_decoding(inputs, model, max_length, temperature:float=1, top_k=50, tokenizer=None):
    assert inputs["input_ids"].shape[0] == 1, "这个最小 demo 只支持 batch_size=1"

    print(f"input: {input}");print()
    if tokenizer is not None:
        print(f"input_decode: {tokenizer.decode(inputs['input_ids'])}"); print()

    for _ in range(max_length):
        output = model(**inputs)
        output_logits = output.logits
        output_logits = output_logits[:, -1, :]

        # temperature
        output_logits = output_logits / temperature

        # top_k
        output_logits_temperature_topk, _ = torch.topk(input=output_logits, k=top_k, dim=-1)
        output_logits[output_logits < output_logits_temperature_topk[0][-1]] = - float('inf')

        # softmax
        probs = F.softmax(input=output_logits, dim=-1)

        # sample
        next_token = torch.multinomial(input=probs, num_samples=1)

        # 组装
        next_attention = torch.ones(inputs['attention_mask'].shape[0], 1)
        inputs['input_ids'] = torch.cat([inputs['input_ids'], next_token], dim=1)
        inputs['attention_mask'] = torch.cat([inputs['attention_mask'], next_attention], dim=1)
        print(f'next_token: {next_token}', end='')
        if tokenizer is not None:
            print(f'next_token_decode: {tokenizer.decode(next_token)}');print()
        if next_token.item() == model.config.eos_token_id:
            break
    return inputs['input_ids']

In [125]:
messages = [
    {'role': 'user', 'content': 'introduce yourself in 10 words'}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    enable_thinking = False,
    add_generation_prompt = True
)

inputs = tokenizer(
    text,
    return_tensors = 'pt'
)

output = sample_decoding(inputs=inputs, model=model, max_length=1000, tokenizer=tokenizer, temperature=1, top_k=60)

output

input: <bound method Kernel.raw_input of <ipykernel.ipkernel.IPythonKernel object at 0x105f532b0>>

input_decode: ['<|im_start|>user\nintroduce yourself in 10 words<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n']

next_token: tensor([[5050]])next_token_decode: ['My']

next_token: tensor([[829]])next_token_decode: [' name']

next_token: tensor([[374]])next_token_decode: [' is']

next_token: tensor([[8515]])next_token_decode: [' Alex']

next_token: tensor([[11]])next_token_decode: [',']

next_token: tensor([[323]])next_token_decode: [' and']

next_token: tensor([[358]])next_token_decode: [' I']

next_token: tensor([[2776]])next_token_decode: ["'m"]

next_token: tensor([[264]])next_token_decode: [' a']

next_token: tensor([[13014]])next_token_decode: [' tech']

next_token: tensor([[60812]])next_token_decode: [' enthusiast']

next_token: tensor([[13]])next_token_decode: ['.']

next_token: tensor([[151645]])next_token_decode: ['<|im_end|>']



tensor([[151644,    872,    198,    396,  47845,   6133,    304,    220,     16,
             15,   4244, 151645,    198, 151644,  77091,    198, 151667,    271,
         151668,    271,   5050,    829,    374,   8515,     11,    323,    358,
           2776,    264,  13014,  60812,     13, 151645]])

In [126]:
tokenizer.decode(output)

["<|im_start|>user\nintroduce yourself in 10 words<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nMy name is Alex, and I'm a tech enthusiast.<|im_end|>"]

## 优缺点
### 优点
- 重塑概率分布, 便于控制输出
- 输出的创造性更强
### 缺点
- 引入了额外的计算开销
- 不确定性增大